# CEP Sim — Brazil
Brand recall simulator for Brazil beer market (Heineken study, N=1,325).

**Brands:** Heineken, Brahma, Budweiser, Corona Extra, Amstel, Skol, and 15 others  
**CEPs:** 11 purchase occasions (Q10–Q20) in Portuguese  
**Config:** `backend/configs/cep_sim_config_brazil.toml`


In [ ]:
import sys
sys.path.insert(0, '../../../..')  # project root

import logging
import pandas as pd
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s - %(message)s')


## 1. Load config

In [ ]:
from pathlib import Path
from backend.analysis.cep_sim.schemas.config import load_cep_sim_config

PROJECT_ROOT = Path('../../../../').resolve()

config = load_cep_sim_config(PROJECT_ROOT / 'backend/configs/cep_sim_config_brazil.toml')

# Resolve relative paths to absolute
config.survey.zip_path      = str(PROJECT_ROOT / config.survey.zip_path)
config.output.outputs_dir   = str(PROJECT_ROOT / config.output.outputs_dir)
config.output.processed_dir = str(PROJECT_ROOT / config.output.processed_dir)

print('Country  :', config.survey.country)
print('ZIP path :', config.survey.zip_path)
print('Outputs  :', config.output.outputs_dir)

## 2. Load raw survey

In [ ]:
from backend.analysis.cep_sim.service.load_data import load_survey, inspect_survey

df = load_survey(config)
info = inspect_survey(df, config)
print(f"{info['row_count']} respondents  |  {info['recall_column_count']} recall columns  |  {len(info['cep_blocks'])} CEP blocks")


## 3. Reshape wide → long

In [ ]:
from backend.analysis.cep_sim.service.reshape_survey import reshape_wide_to_long, save_long_survey

long_df = reshape_wide_to_long(df, config)
save_long_survey(long_df, config)

print(f"Long table : {len(long_df):,} rows")
print(f"Unique CEPs: {long_df['cep_description'].nunique()}")
print(f"Mention rate: {long_df['mentioned'].mean():.1%}")
long_df.head(3)


## 4. Build CEP ontology

In [ ]:
from backend.analysis.cep_sim.service.ontology_builder import build_ontology, save_ontology

cep_master_df, raw_map_df = build_ontology(long_df, config)
save_ontology(cep_master_df, raw_map_df, config)

print(f"{len(cep_master_df)} CEPs across families:")
print(cep_master_df.groupby('cep_family').size().sort_values(ascending=False).to_string())
cep_master_df[['cep_id','cep_family','cep_label']]


## 5. Build respondent memory tables

In [ ]:
from backend.analysis.cep_sim.service.respondent_builder import (
    build_respondents, build_respondent_brand_cep, save_respondent_tables
)

respondents_df = build_respondents(df, config)
rbc_df = build_respondent_brand_cep(long_df, raw_map_df, config)
save_respondent_tables(respondents_df, rbc_df, config)

print(f"{len(respondents_df):,} respondents  |  {len(rbc_df):,} memory edges")
print(f"Segments: {respondents_df['segment'].value_counts().head(5).to_dict()}")

brand_name_map = rbc_df[['brand_id','brand_name']].drop_duplicates().set_index('brand_id')['brand_name'].to_dict()
respondent_ids = respondents_df['respondent_id'].astype(str).tolist()

top = rbc_df.groupby('brand_name').size().sort_values(ascending=False).head(8)
print("\nTop 8 brands by memory edges:")
print(top.to_string())


## Phase 2A/2B — Calibration setup

Holdout split, brand priors (global + CEP-conditioned), parameter fitting, and holdout validation — identical pipeline to UK demo.

In [ ]:
from backend.analysis.cep_sim.service.calibration import (
    make_holdout_split, compute_brand_priors, compute_cep_brand_priors,
    compute_respondent_responsiveness, fit_parameters,
    run_holdout_validation, build_calibration_report,
)
from backend.analysis.cep_sim.service.scenario_library import DEMO_BRAZIL_SCENARIOS

train_ids, holdout_ids = make_holdout_split(respondent_ids)

rbc_train    = rbc_df[rbc_df["respondent_id"].isin(train_ids)].copy()
rbc_holdout  = rbc_df[rbc_df["respondent_id"].isin(holdout_ids)].copy()
long_train   = long_df[long_df["respondent_id"].isin(train_ids)].copy()
long_holdout = long_df[long_df["respondent_id"].isin(holdout_ids)].copy()

brand_priors      = compute_brand_priors(long_train)
cep_brand_priors  = compute_cep_brand_priors(long_train)
responsiveness_map = compute_respondent_responsiveness(rbc_train)

print(f"Train: {len(train_ids)} | Holdout: {len(holdout_ids)}")
print(f"CEP-brand prior entries: {len(cep_brand_priors)}")

In [ ]:
fitted = fit_parameters(
    long_train, rbc_train, cep_master_df, DEMO_BRAZIL_SCENARIOS, config,
    brand_priors=brand_priors, cep_brand_priors=cep_brand_priors,
)
config.defaults.softmax_temperature        = fitted["tau"]
config.defaults.competition_penalty_weight = fitted["gamma"]
config.defaults.base_prior_weight          = fitted["prior_weight"]

print(f"Brazil best params: τ={fitted['tau']}  γ={fitted['gamma']}  β_scale={fitted['prior_weight']}")
print(f"Train MAE: {fitted['mae']:.4f}")

In [ ]:
holdout_results = run_holdout_validation(
    holdout_ids, long_holdout, rbc_holdout, cep_master_df,
    DEMO_BRAZIL_SCENARIOS, config, brand_priors,
    cep_brand_priors=cep_brand_priors,
)

train_results = run_holdout_validation(
    train_ids, long_train, rbc_train, cep_master_df,
    DEMO_BRAZIL_SCENARIOS, config, brand_priors,
    cep_brand_priors=cep_brand_priors,
)

report_md = build_calibration_report(
    train_results, holdout_results, fitted, config,
    n_train=len(train_ids), n_holdout=len(holdout_ids),
)

from pathlib import Path
report_path = Path(config.output.outputs_dir) / "calibration_report.md"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(report_md)

print(f"Holdout MAE: {holdout_results['mae']:.4f}  |  mean ρ: {holdout_results['mean_rho']:.4f}")
print(f"Report saved: {report_path}")

## 6. Baseline recall — single respondent

In [ ]:
from backend.analysis.cep_sim.service.recall_engine import get_recall_probs, rank_brands, BRAZIL_SCENARIOS

SCENARIOS = BRAZIL_SCENARIOS
sample_id = str(respondents_df['respondent_id'].iloc[0])
scenario  = SCENARIOS[0]  # trendy_bar

probs  = get_recall_probs(sample_id, scenario['active_ceps'], rbc_df, cep_master_df, config)
ranked = rank_brands(probs)

print(f"Respondent {sample_id} | Scenario: {scenario['scenario_name']}")
print(f"{'Rank':<5} {'Brand':<25} {'Prob':>6}")
for rank, (brand_id, prob) in enumerate(ranked[:8], 1):
    print(f"{rank:<5} {brand_name_map.get(brand_id, brand_id):<25} {prob:.4f}")


## 7. Baseline recall — all respondents × all scenarios

In [ ]:
from backend.analysis.cep_sim.service.validator import run_scenario_recall

scenario_recall_df = run_scenario_recall(
    respondent_ids, SCENARIOS, rbc_df, cep_master_df, brand_name_map, config
)
print(f"Scenario recall: {len(scenario_recall_df):,} rows")

# Average recall prob — trendy bar
trendy = scenario_recall_df[scenario_recall_df['scenario_name'] == 'trendy_bar']
avg    = trendy.groupby('brand_name')['recall_prob'].mean().sort_values(ascending=False).head(8)

fig, ax = plt.subplots(figsize=(9, 4))
avg.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Brazil — Avg Recall Probability: Trendy Bar')
ax.set_ylabel('Avg Recall Prob')
plt.xticks(rotation=35, ha='right'); plt.tight_layout(); plt.show()


## 8. Calibration check

In [ ]:
from backend.analysis.cep_sim.service.validator import run_calibration_check

cal_df = run_calibration_check(scenario_recall_df, long_df)
print(f"MAE: {cal_df.attrs['mae']:.4f}  (mean |predicted recall_prob − observed mention rate|)")
cal_df


## 9. Define Heineken ad and apply it

In [ ]:
from backend.analysis.cep_sim.service.ad_engine import Ad, apply_ad_to_population, save_episodic_events

# Resolve CEP IDs
trendy_bar_cep = cep_master_df[
    cep_master_df['cep_description'].str.contains('bar da moda', case=False, na=False)
]['cep_id'].values[0]

outdoor_cep = cep_master_df[
    cep_master_df['cep_description'].str.contains('ao ar livre em um dia quente', case=False, na=False)
]['cep_id'].values[0]

heineken_ad = Ad(
    ad_id='ad_heineken_brazil_trendy_bar_001',
    brand_id='brand_heineken',
    brand_name='Heineken',
    focal_ceps=[trendy_bar_cep],
    secondary_ceps=[outdoor_cep],
    branding_clarity=0.9,
    attention_weight=1.0,
    channel='social_media',
    emotion='confidence',
)

rbc_post, events = apply_ad_to_population(respondent_ids, heineken_ad, rbc_df, config)
save_episodic_events(events, config)

print(f"Ad: {heineken_ad.ad_id}")
print(f"Focal CEP   : {trendy_bar_cep}")
print(f"Secondary   : {outdoor_cep}")
print(f"Applied to  : {len(events)} respondents")
print(f"Edges pre   : {len(rbc_df):,}")
print(f"Edges post  : {len(rbc_post):,}")


## 10. Before vs after recall

In [ ]:
from backend.analysis.cep_sim.service.validator import run_ad_impact, build_segment_summary

ad_impact_df = run_ad_impact(
    respondent_ids, SCENARIOS, rbc_df, rbc_post, cep_master_df, brand_name_map, config
)

impact = (
    ad_impact_df[ad_impact_df['scenario_name'] == 'trendy_bar']
    .groupby('brand_name')[['recall_pre','recall_post','delta']]
    .mean().round(4)
    .sort_values('delta', ascending=False)
    .head(8)
)
print("Brazil — Avg recall shift: Trendy Bar (Heineken ad)")
print(impact.to_string())


In [ ]:
top8 = impact.head(8).index
plot_df = impact.loc[top8][['recall_pre','recall_post']]

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(plot_df))
ax.bar([i - 0.2 for i in x], plot_df['recall_pre'],  width=0.4, label='Pre',  color='steelblue')
ax.bar([i + 0.2 for i in x], plot_df['recall_post'], width=0.4, label='Post', color='coral')
ax.set_xticks(list(x)); ax.set_xticklabels(plot_df.index, rotation=35, ha='right')
ax.set_title('Brazil — Recall Pre vs Post: Trendy Bar (Heineken ad)')
ax.set_ylabel('Avg Recall Score'); ax.legend()
plt.tight_layout(); plt.show()


## 11. Segment summary

In [ ]:
segment_summary_df = build_segment_summary(ad_impact_df, respondents_df)
print(f"Segment summary: {len(segment_summary_df):,} rows")
segment_summary_df[segment_summary_df['scenario_name'] == 'trendy_bar'].head(12)


## 12. Export + sanity checks

In [ ]:
from backend.analysis.cep_sim.service.validator import save_outputs, run_sanity_checks

paths = save_outputs(scenario_recall_df, ad_impact_df, segment_summary_df, config)
for name, path in paths.items():
    print(f"  {name}: {path}")

print("\n=== Sanity Checks ===")
for k, v in run_sanity_checks(scenario_recall_df, ad_impact_df).items():
    print(f"  {k}: {v}")
